# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
import importlib, agents.deals
importlib.reload(agents.deals)
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-6-luna'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 5/5 [00:18<00:00,  3.65s/it]


In [3]:
len(deals)

46

In [4]:
deals[10].describe()

'Title: onn Core 7" 64GB Android 16 Tablet (2026) for $87 + free shipping\nDetails: Walmart offers the onn Core 7" 64GB Android 16 Tablet for $87.\xa0 That\'s a $10 low.\xa0 Shipping is free. Buy Now at Walmart\nFeatures: 7" LCD touchscreen, 1024 x 600 resolution Octa-core 2.0 GHz processor w/ 4GB RAM 64GB storage w/ microSD card support 2MP front and 2MP rear cameras Wi-Fi 5 & Bluetooth 5.0; no cellular support Up to 10 hours of battery life\nURL: https://www.dealnews.com/onn-Core-7-64-GB-Android-16-Tablet-2026-for-87-free-shipping/22412619.html?iref=rss-c39'

### We are going to ask GPT-6 Luna to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [9]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Lewitt LCT 040 MATCH Small-Diaphragm Condenser Microphone Stereo Pair for $199 + free shipping
Details: Sweetwater has the Lewitt LCT 040 MATCH stereo pair at $199, down from $259, through January 7, 2027. That is $60 off for two matched condenser microphones. Shipping is free, and Sweetwater adds a free 2-year warranty. Buy Now at Sweetwater
Features: Stereo pair of two small-

In [10]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="none")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='A 16-inch WQXGA touchscreen laptop with an Intel Core Ultra 7 256V processor and integrated Intel Arc graphics. It includes 32GB of RAM, a 1TB SSD, and Windows 11 Home, along with a fingerprint reader and backlit keyboard. This refurbished Dell 16 Plus is finished in Ice Blue.', price=684.0, url='https://vipoutlet.com/products/dell-db16250-dell-16-plus-db16250-16-wqxga-touchscreen-ultra-7-256v-2-2ghz-intel-arc-graphics-32gb-ram-1tb-ssd-win-11-home-ice-blue'), Deal(product_description='The ASUS TUF Gaming A16 is a 16-inch WUXGA laptop powered by an AMD Ryzen 7 7445HS processor and NVIDIA GeForce RTX 4050 graphics with 6GB of dedicated memory. It has 16GB of RAM, a 512GB SSD, and Windows 11 Home. Other features include a backlit keyboard and a Mecha Gray finish.', price=679.0, url='https://vipoutlet.com/products/asus-fa607nug-wh73-d-tuf-gaming-a16-16-wuxga-ryzen-7-7445hs-3-2ghz-nvidia-geforce-rtx-4050-6gb-16gb-ram-512gb-ssd-win-11-home-mecha

In [11]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


A 16-inch WQXGA touchscreen laptop with an Intel Core Ultra 7 256V processor and integrated Intel Arc graphics. It includes 32GB of RAM, a 1TB SSD, and Windows 11 Home, along with a fingerprint reader and backlit keyboard. This refurbished Dell 16 Plus is finished in Ice Blue.
684.0
https://vipoutlet.com/products/dell-db16250-dell-16-plus-db16250-16-wqxga-touchscreen-ultra-7-256v-2-2ghz-intel-arc-graphics-32gb-ram-1tb-ssd-win-11-home-ice-blue

The ASUS TUF Gaming A16 is a 16-inch WUXGA laptop powered by an AMD Ryzen 7 7445HS processor and NVIDIA GeForce RTX 4050 graphics with 6GB of dedicated memory. It has 16GB of RAM, a 512GB SSD, and Windows 11 Home. Other features include a backlit keyboard and a Mecha Gray finish.
679.0
https://vipoutlet.com/products/asus-fa607nug-wh73-d-tuf-gaming-a16-16-wuxga-ryzen-7-7445hs-3-2ghz-nvidia-geforce-rtx-4050-6gb-16gb-ram-512gb-ssd-win-11-home-mecha-gray

This TCL Android tablet features a 12.2-inch 2.4K NXTPAPER display with a refresh rate of up to 

In [12]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 46 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [15]:
result

DealSelection(deals=[Deal(product_description='This Dell 16 Plus laptop has a 16-inch WQXGA touchscreen, an Intel Core Ultra 7 256V processor, integrated Intel Arc graphics, 32GB of RAM, and a 1TB SSD. It runs Windows 11 Home and includes a webcam, fingerprint reader, and backlit keyboard. The unit is refurbished and finished in Ice Blue.', price=684.0, url='https://vipoutlet.com/products/dell-db16250-dell-16-plus-db16250-16-wqxga-touchscreen-ultra-7-256v-2-2ghz-intel-arc-graphics-32gb-ram-1tb-ssd-win-11-home-ice-blue'), Deal(product_description='The ASUS TUF Gaming A16 is a 16-inch WUXGA laptop powered by an AMD Ryzen 7 7445HS processor and an NVIDIA GeForce RTX 4050 with 6GB of video memory. It has 16GB of RAM, a 512GB SSD, and Windows 11 Home. Additional features include a backlit keyboard and webcam; this unit is refurbished.', price=679.0, url='https://vipoutlet.com/products/asus-fa607nug-wh73-d-tuf-gaming-a16-16-wuxga-ryzen-7-7445hs-3-2ghz-nvidia-geforce-rtx-4050-6gb-16gb-ram-512

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [24]:
load_dotenv(override=True)

True

In [25]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [26]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [27]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [28]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [29]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [30]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message


20:54:30 - LiteLLM:INFO: utils.py:4336 - 
LiteLLM completion() model= claude-haiku-5-5; provider = anthropic


INFO:LiteLLM:
LiteLLM completion() model= claude-haiku-5-5; provider = anthropic


20:54:31 - LiteLLM:INFO: utils.py:1752 - Wrapper: Completed Call, calling success_handler


INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
